In [ ]:
#外れ値処理
# --- 1. 必要なライブラリのインポート ---
import numpy as np
import pandas as pd

# --- 2. データの読み込み ---
# ※ kaggle-snippets から titanic フォルダのデータを参照する場合の相対パス
df = pd.read_csv("../titanic/train.csv")
df_test = pd.read_csv("../titanic/test.csv")

#年齢の外れ値をnullにする（上のグラフから右に引きずる形だと分かった）
df.loc[df['Age'] > df['Age'].quantile(0.9), 'Age'] = np.nan
df_test.loc[df_test['Age'] > df['Age'].quantile(0.9), 'Age'] = np.nan

# df['Age'].quantile(0.9)
quantile(0.9) は「下から数えて 90% 目（＝上位 10%）の位置にある数値」を計算します。

# df.loc[df['Age'] > 境界線, 'Age'] = np.nan
df['Age'] > 境界線: 年齢が 50歳 より大きい人（上位 10% の高年齢層）を探します。


np.nan: その人たちの Age の値を、NumPy の「空データ（NaN）」に上書きして消してしまいます。

df.loc[df['Age'] > 50, 'Age'] = np.nan

       └────①────┘    └─②─┘    └──③──┘
       
① 行の条件: 「年齢が50歳より上の人（行）」を選んで

② 列の指定: 「Age（年齢）の列」だけをターゲットにして

③ 変更: 「np.nan（空っぽ）」に書き換える！

テストデータ（df_test）の処理をするときも、訓練データ（df）で作ったのと同じ基準（例: 50歳）を使って揃える

In [ ]:
#年齢の外れ値を平均で埋める

# --- 1. 必要なライブラリのインポート ---
import numpy as np
import pandas as pd

# --- 2. データの読み込み ---
# ※ kaggle-snippets から titanic フォルダのデータを参照する場合の相対パス
df = pd.read_csv("../titanic/train.csv")
df_test = pd.read_csv("../titanic/test.csv")

age = df['Age'].mean()
df.loc[df['Age'].isna(), 'Age'] = age
df_test.loc[df_test['Age'].isna(), 'Age'] = age

# df.loc[df['Age'].isna(), 'Age'] = age

Age が NaN（空っぽ）になっている行」を探します。

# df.loc[..., 'Age'] = age
その空っぽになっている場所だけに、さっき計算した平均値（29.7歳）を上書きして埋めます。

# df_test.loc[df_test['Age'].isna(), 'Age'] = age
テストデータ（df_test）の空っぽ部分も、訓練データから計算した平均値（age ＝ 29.7歳）を使って埋めています。

ここでもテストデータ自身の平均値ではなく、訓練データの平均値を使うのがデータ分析の共通ルール（データリーク防止）です！

In [1]:
#使わない要素（カラム）を消去
# --- 1. 必要なライブラリのインポート ---
import numpy as np
import pandas as pd

# --- 2. データの読み込み ---
# ※ kaggle-snippets から titanic フォルダのデータを参照する場合の相対パス
df = pd.read_csv("../titanic/train.csv")
df_test = pd.read_csv("../titanic/test.csv")

category_list = ['SibSp','Parch','Fare', 'Cabin', 'Embarked','Ticket','Name']

df.drop(category_list, axis=1, inplace=True)
df_test.drop(category_list, axis=1, inplace=True)

# category_list
1行目で作った「消したい列のリスト」を指定しています。

# axis=1
「列（タテの列）」を消す という指定です。
(※ axis=0 だと「行（ヨコの一行）」を消す意味になります)

# inplace=True
元のデータフレーム（df）の中身を直接書き換えて上書き更新する という指定です。
(※ これを書かないと、削除された結果が表示されるだけで元の df は変わらないままになります)

Name / Ticket / Cabin: 文字列（テキストデータ）のままだと機械学習モデルに入力できないため、高度な前処理（特徴量抽出）をしない段階では一旦除外する。

SibSp / Parch / Fare / Embarked: 今回のモデルでは「年齢（Age）などのシンプルな要素だけで予測を試す」という実験にするために、使わない列として除外している。

In [ ]:
#df確認用
df.head()

,PassengerId,Survived,Pclass,Sex,Age
0,1,0,3,male,22.0
1,2,1,1,female,38.0
2,3,1,3,female,26.0
3,4,1,1,female,35.0
4,5,0,3,male,35.0


In [3]:
#ラベルエンコーディング
# --- 1. 必要なライブラリのインポート ---
from sklearn.preprocessing import OrdinalEncoder

# --- 2. データの読み込み ---
# ※ kaggle-snippets から titanic フォルダのデータを参照する場合の相対パス
df = pd.read_csv("../titanic/train.csv")
df_test = pd.read_csv("../titanic/test.csv")


label_encoder = OrdinalEncoder()
df[['Sex']] = label_encoder.fit_transform(df[['Sex']].values)
df_test[['Sex']] = label_encoder.transform(df_test[['Sex']].values)

性別は２種類しかないからラベル（順序）を考えなくていい

In [4]:
# --- 1. 必要なライブラリのインポート ---
import numpy as np
import pandas as pd

# --- 2. データの読み込み ---
# ※ kaggle-snippets から titanic フォルダのデータを参照する場合の相対パス
df = pd.read_csv("../titanic/train.csv")
df_test = pd.read_csv("../titanic/test.csv")

#X,yに目的，説明変数代入
X = df.iloc[0:, 2:].values
y = df.iloc[:, 1].values

X_test = df_test.iloc[:, 1:].values

X.shape

(891, 10)

# df.iloc[0:, 2:]

行：0: ➔ 0行目から最後まで（全データ行）

列：2: ➔ 2列目から一番右の列まで

※タイタニックの train.csv は通常 0列目が PassengerId（乗客ID）、1列目が Survived（生存フラグ） になっているため、2列目以降（Pclass, Sex, Age など）のヒント部分だけを切り出しています。

# .values

Pandas の表（Dataframe）から見出し（列名）を外して、計算用の数値の集まり（NumPy配列） に変換します。

# df.iloc[:, 1]

行：: ➔ 全行

列：1 ➔ 1列目（Survived：生存したかどうかの 0 or 1）だけを指定

これで y には AIに予測させたい「正解ラベル（0か1）」 だけが抽出されます。

# X_test = df_test.iloc[:, 1:].values

テストデータ（df_test）には、当たり前ですが正解データである Survived（1列目）が入っていません。

0列目が PassengerId なので、1:（1列目以降のすべて）を指定して、モデルに入力するための説明変数だけを切り出しています。

# X.shape

切り出した X が 「タテ何行 × ヨコ何列 の行列になっているか」 を確認する命令です。

In [5]:
# 1. 訓練データ（df）をクリーンデータとして保存
df.to_csv("train_clean.csv", index=False)

# 2. テストデータ（df_test）をクリーンデータとして保存
df_test.to_csv("test_clean.csv", index=False)

print("クリーン CSV の作成が完了しました！")

クリーン CSV の作成が完了しました！


# 敬称（Title）の抽出とカテゴリ統合
Name 列の正規表現抽出で敬称を取り出し、少数の敬称（Mlle等）や稀な敬称を性別に基づいて Mr, Miss, Mrs, Master の4つに標準化します。

In [2]:
import pandas as pd
import numpy as np
import joblib

# データのロード（例）
train = pd.read_csv('../titanic/train.csv')

df = train.copy().set_index('PassengerId')

# 正規表現で敬称を抽出
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# 表記揺れの統一
title_mapping = {'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'}
df['Title'] = df['Title'].replace(title_mapping)

# 4大敬称以外の例外を性別（Sex）から判定して補正
title_mask = ~df['Title'].isin(['Mr', 'Miss', 'Mrs', 'Master'])
df.loc[title_mask, 'Title'] = df.loc[title_mask, 'Sex'].map({'male': 'Mr', 'female': 'Mrs'})

# 欠損値処理（Age, Embarked, Fare）
Age: 敬称ごとの年齢中央値（集計済み数値）を使って補完


Embarked: 最頻値（mode）で補完


Fare: 中央値（median）で補完

In [3]:
import pandas as pd
import numpy as np
import joblib

# データのロード（例）
train = pd.read_csv('../titanic/train.csv')

# 敬称別の年齢中央値マップ
title_age_medians = {
    'Mr': 32.32,
    'Miss': 21.68,
    'Mrs': 35.86,
    'Master': 4.57
}

# 年齢の欠損値を敬称別に埋める
for title, median_age in title_age_medians.items():
    age_mask = (df['Age'].isnull()) & (df['Title'] == title)
    df.loc[age_mask, 'Age'] = median_age

# Embarked と Fare の欠損値補完
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# 新規特徴量の作成
相互作用特徴量 
(Age*Class, Age*Fare)

同乗家族数 (FamilySize, IsAlone)

年齢・運賃のビン分け (AgeBand, FareBand)

運賃の対数変換 (Fare_log): 歪んだ分布を正規分布に近づける

In [4]:
import pandas as pd
import numpy as np
import joblib

# データのロード（例）
train = pd.read_csv('../titanic/train.csv')

# 掛け合わせ特徴量
df['Age*Class'] = df['Age'] * df['Pclass']
df['Age*Fare'] = df['Age'] * df['Fare']

# 家族関連
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# ビニング（カテゴリ化）
df['AgeBand'] = pd.cut(df['Age'], bins=[0, 12, 20, 40, 60, np.inf], labels=[0, 1, 2, 3, 4]).astype(int)
df['FareBand'] = pd.qcut(df['Fare'], q=4, labels=[0, 1, 2, 3]).astype(int)

# 対数変換（右に裾が長い分布の歪み対策）
df['Fare_log'] = np.log1p(df['Fare'])

# ワンホットエンコーディングと不要項目の削除
カテゴリ変数を pd.get_dummies でダミー変数化し、元のカテゴリ列や加工済みの不要列を削除します。

In [5]:
import pandas as pd
import numpy as np
import joblib

# データのロード（例）
train = pd.read_csv('../titanic/train.csv')

# ワンホットエンコーディング
df_sex = pd.get_dummies(df['Sex'], prefix='sex', drop_first=True, dtype=int)
df_Pclass = pd.get_dummies(df['Pclass'], prefix='class', drop_first=True, dtype=int)
df_Embarked = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True, dtype=int)
df_Title = pd.get_dummies(df['Title'], prefix='Title', drop_first=False, dtype=int)

# 結合
df = pd.concat([df, df_sex, df_Pclass, df_Embarked, df_Title], axis=1)

# 元データ・不要列の削除
drop_cols = ['Sex', 'Pclass', 'Name', 'Ticket', 'Embarked', 'Cabin', 'Title', 'Fare', 'SibSp', 'Parch']
df = df.drop(drop_cols, axis=1)

# 数値特徴量の標準化（Zスコア標準化）
数値型のカラムに対して平均0、分散1になるよう標準化を行います。

In [7]:
import pandas as pd
import numpy as np
import joblib

# データのロード（例）
train = pd.read_csv('../titanic/train.csv')

numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns

mew = df[numeric_columns].mean(axis=0)
std = df[numeric_columns].std(axis=0)

df[numeric_columns] = (df[numeric_columns] - mew) / std

# 再利用のための前処理関数まとめ
実験・検証が終わったら、本番用に1つの関数に集約しておくと推論（test.csv）時にも同じ前処理を適用できて便利です。

In [ ]:
import pandas as pd
import numpy as np
import joblib

# データのロード（例）
train = pd.read_csv('../titanic/train.csv')


numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns

mew = df[numeric_columns].mean(axis=0)
std = df[numeric_columns].std(axis=0)

df[numeric_columns] = (df[numeric_columns] - mew) / std